Prediction of Clinical Trial Attrition — Modeling & Geographic Analysis
HIDS-6001, Massive Health Data Fundamentals, Fall 2024
Team 3a: Stephanie Araki, Pari Shah, Samantha Zocher

Encodes attrition rate as a binary label (high vs. low, split at the sample median),
trains and compares three classifiers, extracts feature importance via XGBoost, and
maps trial facilities geographically by attrition rate using Folium.

Reported results (final project report):
  Random Forest: accuracy 0.694, recall 0.685, precision 0.690, F1 0.687, AUC-ROC 0.768
  Logistic Regression: AUC-ROC 0.691   |   SVM: AUC-ROC 0.634

In [ ]:
import folium
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

In [ ]:
RANDOM_STATE = 42

In [ ]:
NUMERIC_FEATURES = [
    "duration_years", "actual_enrollment", "locations_count", "adverse_event_count",
    "days_since_start", "min_age_eligible",
]

In [ ]:
CATEGORICAL_FEATURES = [
    "organization_class_refined", "primary_purpose", "condition_category",
    "ruca_binary_most_common", "phase",
]

### `encode_labels`

In [ ]:
def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    """High attrition (1) = dropout % above the sample median; low (0) = at or below."""
    median_dropout = df["dropout_percentage_all"].median()
    df["dropout_label"] = (df["dropout_percentage_all"] > median_dropout).astype(int)
    return df

### `build_feature_matrix`

In [ ]:
def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode categorical fields and combine with already-numeric features."""
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    cat_encoded = encoder.fit_transform(df[CATEGORICAL_FEATURES].fillna("Unknown"))
    cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(CATEGORICAL_FEATURES))

    numeric_df = df[NUMERIC_FEATURES].fillna(df[NUMERIC_FEATURES].median())
    return pd.concat([numeric_df.reset_index(drop=True), cat_df.reset_index(drop=True)], axis=1)

### `train_and_evaluate`

In [ ]:
def train_and_evaluate(X: pd.DataFrame, y: pd.Series):
    """Train/test split (80/20) and compare Logistic Regression, Random Forest, and SVM."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
        "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
        results[name] = {
            "accuracy": accuracy_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "auc_roc": roc_auc_score(y_test, probs),
        }
    return results, (X_train, X_test, y_train, y_test)

### `xgboost_feature_importance`

In [ ]:
def xgboost_feature_importance(X_train, y_train, X_test, y_test):
    """XGBoost feature importance highlighted enrollment size, trial duration,
    location count, and adverse event count as the top attrition drivers."""
    model = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    return importances.sort_values(ascending=False)

### `build_attrition_map`

In [ ]:
def build_attrition_map(df: pd.DataFrame, out_html: str = "attrition_map.html"):
    """Plot trial facility locations on a Folium map, color-coded by high (red) vs.
    low (green) attrition, to visualize geographic attrition hotspots."""
    m = folium.Map(location=[39.8, -98.6], zoom_start=4)  # centered on continental US
    for _, row in df.dropna(subset=["latitude", "longitude"]).iterrows():
        color = "red" if row["dropout_label"] == 1 else "green"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3, color=color, fill=True, fill_opacity=0.6,
        ).add_to(m)
    m.save(out_html)
    return m

### Run

In [ ]:
if __name__ == "__main__":
    df = pd.read_csv("data/nct_features_all.csv")
    df = encode_labels(df)

    X = build_feature_matrix(df)
    y = df["dropout_label"]

    results, splits = train_and_evaluate(X, y)
    print("Model comparison:")
    for name, metrics in results.items():
        print(f"  {name}: {metrics}")

    _, X_test, y_train, y_test = splits[1], splits[1], splits[2], splits[3]
    top_features = xgboost_feature_importance(splits[0], splits[2], splits[1], splits[3])
    print("\nTop attrition drivers (XGBoost feature importance):")
    print(top_features.head(10))

    build_attrition_map(df)